# Kaggle 2xT4 Full Repro

Notebook ini menjalankan semuanya lewat notebook:
1. Clone repo ke /kaggle/temp
2. Setup .venv dan install requirements.txt
3. Download parent teacher model dari Hugging Face
4. Gabung metadata.csv + metadata_indsp.csv
5. Prepare dataset Arrow
6. Setup W&B hardcoded key
7. Distillation training di 2x T4

Semua command Python dijalankan via: uv run --python .venv/bin/python

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# ================= User Config =================
REPO_URL = "https://github.com/AneKazek/malesbgt.git"
REPO_BRANCH = "main"

WANDB_API_KEY = "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"
WANDB_ENTITY = "haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember"
WANDB_PROJECT = "kaceve"

HF_REPO_ID = "Eempostor/F5-TTS-INDO-FINETUNE-V2"
HF_CKPT_FILENAME = "f5_tts_indo_v2.pt"

CSV_1 = Path("/kaggle/input/datasets/benedictusryugunawan/tts-indo/data/metadata.csv")
# CSV_2 = Path("/kaggle/input/datasets/benedictusryugunawan/tts-indo/data/metadata_indsp.csv")  # dimatikan sementara
CSV_2 = None

WORKDIR = Path("/kaggle/temp")
REPO_DIR = WORKDIR / "kcv-tts"
VENV_DIR = REPO_DIR / ".venv"
VENV_PY = VENV_DIR / "bin/python"

HF_OUT_DIR = REPO_DIR / "ckpts/hf/Eempostor_F5-TTS-INDO-FINETUNE-V2"
TEACHER_CKPT = HF_OUT_DIR / HF_CKPT_FILENAME
MERGED_CSV = REPO_DIR / "data/metadata_merged.csv"
PREPARED_DATASET_DIR = REPO_DIR / "data/datasetku_pinyin"

def run_cmd(cmd, cwd=None, env=None, timeout=None):
    printable = cmd if isinstance(cmd, str) else " ".join(str(x) for x in cmd)
    print("\n$", printable)
    if timeout is not None:
        print(f"(timeout={timeout}s)")
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
        text=True,
        timeout=timeout,
    )

def run_py(args, cwd=None, env=None, timeout=None):
    # --no-sync mencegah uv mengubah isi .venv saat eksekusi python command.
    return run_cmd(["uv", "run", "--no-sync", "--python", str(VENV_PY), "python", *args], cwd=cwd, env=env, timeout=timeout)

print("Config siap.")
print("CSV_1 exists:", CSV_1.exists())
print("CSV_2 enabled:", CSV_2 is not None)
if CSV_2 is not None:
    print("CSV_2 exists:", CSV_2.exists())

In [ ]:
# 1) Clone repo ke /kaggle/temp
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

run_cmd(["git", "clone", "--recursive", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])
run_cmd(["ls", "-lah", str(REPO_DIR)])

In [ ]:
# 2) Setup .venv + install dependency profile (local/kaggle parity)
if shutil.which("uv") is None:
    run_cmd(["python3", "-m", "pip", "install", "-U", "uv"])

TARGET_PY_MM = "3.11"
TARGET_PY = Path(f"/usr/bin/python{TARGET_PY_MM}")

run_cmd(["apt-get", "update", "-y"])
run_cmd([
    "apt-get",
    "install",
    "-y",
    f"python{TARGET_PY_MM}",
    f"python{TARGET_PY_MM}-venv",
    f"python{TARGET_PY_MM}-dev",
    "build-essential",
])

if not TARGET_PY.exists():
    raise FileNotFoundError(f"Interpreter target tidak ditemukan: {TARGET_PY}")

run_cmd(["uv", "venv", "--python", str(TARGET_PY), "--clear", str(VENV_DIR)])
run_cmd([str(VENV_PY), "-c", "import sys; print('venv python =', sys.version); assert sys.version_info[:2] == (3, 11)"])

# Setuptools<82 menghindari masalah build extension tertentu (mamba/causal-conv1d).
run_cmd([
    "uv",
    "pip",
    "install",
    "--python",
    str(VENV_PY),
    "--upgrade",
    "pip",
    "wheel",
    "setuptools<82",
])

req_candidates = [
    REPO_DIR / "requirements-torch28-cu12-localmatch.txt",
    REPO_DIR / "requirements-kaggle-torch210.txt",
]

REQ_PROFILE = next((p for p in req_candidates if p.exists()), None)
if REQ_PROFILE is None:
    raise FileNotFoundError("Tidak menemukan file profile requirements untuk setup training.")

print("Using dependency profile:", REQ_PROFILE)
run_cmd([
    "uv",
    "pip",
    "install",
    "--python",
    str(VENV_PY),
    "--index-strategy",
    "unsafe-best-match",
    "-r",
    str(REQ_PROFILE),
])

# src layout butuh editable install agar import f5_tts stabil di semua cell script/module.
run_cmd([
    "uv",
    "pip",
    "install",
    "--python",
    str(VENV_PY),
    "-e",
    str(REPO_DIR),
], cwd=REPO_DIR)

# Final guard: pastikan stack torch sesuai target training mamba.
run_cmd([
    "uv",
    "pip",
    "install",
    "--python",
    str(VENV_PY),
    "--index-url",
    "https://download.pytorch.org/whl/cu128",
    "--extra-index-url",
    "https://pypi.org/simple",
    "--index-strategy",
    "unsafe-best-match",
    "--force-reinstall",
    "--no-cache-dir",
    "torch==2.8.0+cu128",
    "torchvision==0.23.0+cu128",
    "torchaudio==2.8.0+cu128",
    "nvidia-nccl-cu12==2.27.3",
    "nvidia-nvjitlink-cu12==12.8.93",
])

run_cmd([
    str(VENV_PY),
    "-c",
    "import torch, torchvision, torchaudio, setuptools, f5_tts; "
    "print('torch =', torch.__version__, 'cuda =', torch.version.cuda); "
    "print('torchvision =', torchvision.__version__); "
    "print('torchaudio =', torchaudio.__version__); "
    "print('setuptools =', setuptools.__version__); "
    "print('f5_tts import = ok')"
])

In [ ]:
# 3) Validasi GPU di Kaggle
run_py([
    "-c",
    "import torch; print('torch', torch.__version__); print('cuda_count', torch.cuda.device_count()); [print(i, torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())]",
])

In [ ]:
# 4) Download parent teacher model dari HF
run_cmd(["uv", "pip", "install", "--python", str(VENV_PY), "-U", "huggingface_hub"])

download_script = "\n".join([
    "from pathlib import Path",
    "from huggingface_hub import hf_hub_download",
    f"repo_id = {HF_REPO_ID!r}",
    f"filename = {HF_CKPT_FILENAME!r}",
    f"out_dir = Path(r'{HF_OUT_DIR}')",
    "out_dir.mkdir(parents=True, exist_ok=True)",
    "path = hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(out_dir), local_dir_use_symlinks=False)",
    "print(path)",
])

run_py(["-c", download_script], cwd=REPO_DIR)
run_cmd(["ls", "-lah", str(HF_OUT_DIR)])

In [ ]:
# 5) Merge 2 CSV metadata -> audio_file|text (pipe-delimited)
import csv
import pandas as pd

def _normalize_metadata_df(df: pd.DataFrame, path: Path) -> pd.DataFrame:
    df.columns = [str(c).strip() for c in df.columns]

    audio_candidates = ["audio_file", "audio_path", "wav_path", "path", "file"]
    text_candidates = ["text", "transcript", "sentence", "normalized_text", "utterance"]

    audio_col = next((c for c in audio_candidates if c in df.columns), None)
    text_col = next((c for c in text_candidates if c in df.columns), None)

    if audio_col is None or text_col is None:
        raise ValueError(f"Kolom tidak cocok di {path}. Dapat: {list(df.columns)}")

    out = df[[audio_col, text_col]].copy()
    out.columns = ["audio_file", "text"]
    out["audio_file"] = out["audio_file"].astype(str).str.strip()
    out["text"] = out["text"].astype(str).str.strip()
    out = out[(out["audio_file"] != "") & (out["text"] != "")]

    source_dir = path.parent
    prefer_indsp = "indsp" in path.name.lower()

    def absolutize(p: str) -> str:
        raw = str(p).strip().strip('"').strip("'")
        pp = Path(raw).expanduser()
        if pp.is_absolute():
            return str(pp)

        has_dir = ("/" in raw) or ("\\" in raw)
        candidates = [source_dir / pp]

        # metadata_indsp sering berisi filename polos (tanpa folder), jadi coba prefix indsp/.
        if not has_dir:
            if prefer_indsp:
                candidates.insert(0, source_dir / "indsp" / pp)
            else:
                candidates.append(source_dir / "wavs" / pp)
                candidates.append(source_dir / "indsp" / pp)

        for cand in candidates:
            if cand.exists():
                return str(cand.resolve())

        return str(candidates[0].resolve())

    out["audio_file"] = out["audio_file"].map(absolutize)
    out["source_csv"] = str(path)
    return out

def _manual_parse_metadata(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8-sig", errors="replace") as f:
        first_non_empty = ""
        for ln in f:
            if ln.strip():
                first_non_empty = ln
                break
        f.seek(0)

        delim = "|" if first_non_empty.count("|") >= first_non_empty.count(",") else ","

        for i, ln in enumerate(f):
            ln = ln.strip()
            if not ln:
                continue
            if i == 0 and "audio_file" in ln.lower() and "text" in ln.lower():
                continue
            if delim not in ln:
                continue
            audio, text = ln.split(delim, 1)
            audio = audio.strip().strip('"')
            text = text.strip()
            if audio and text:
                rows.append((audio, text))

    df = pd.DataFrame(rows, columns=["audio_file", "text"])
    df["source_csv"] = str(path)
    return df

def load_metadata(path: Path | None) -> pd.DataFrame:
    if path is None:
        print("load_metadata: path None, skip.")
        return pd.DataFrame(columns=["audio_file", "text", "source_csv"])
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Metadata CSV tidak ditemukan: {path}")

    parse_attempts = [
        dict(
            sep="|",
            engine="python",
            dtype=str,
            keep_default_na=False,
            encoding="utf-8-sig",
            on_bad_lines="skip",
            quoting=csv.QUOTE_NONE,
        ),
        dict(
            sep=",",
            engine="python",
            dtype=str,
            keep_default_na=False,
            encoding="utf-8-sig",
            on_bad_lines="skip",
            quoting=csv.QUOTE_NONE,
        ),
    ]

    for kwargs in parse_attempts:
        try:
            df = pd.read_csv(path, **kwargs)
            out = _normalize_metadata_df(df, path)
            if len(out) > 0:
                return out
        except Exception:
            pass

    print(f"Parser fallback aktif untuk {path}")
    return _normalize_metadata_df(_manual_parse_metadata(path), path)

sources = [CSV_1]
if CSV_2 is not None:
    sources.append(CSV_2)
else:
    print("CSV_2/metadata_indsp dimatikan sementara. Hanya pakai CSV_1.")

dfs = [load_metadata(path) for path in sources]
merged = pd.concat(dfs, ignore_index=True).drop_duplicates(subset=["audio_file", "text"])

exists_mask = merged["audio_file"].map(lambda p: Path(p).exists())
missing = int((~exists_mask).sum())
if missing:
    print(f"Dropping {missing} rows with missing audio paths.")
merged = merged[exists_mask].copy()

MERGED_CSV.parent.mkdir(parents=True, exist_ok=True)
merged[["audio_file", "text"]].to_csv(MERGED_CSV, sep="|", index=False)

print("Merged rows (after drop missing):", len(merged))
print("Saved:", MERGED_CSV)
print("Source CSVs:", *sources)
print(merged[["audio_file", "text"]].head(5))

In [ ]:
# 6) Siapkan data/Emilia_ZH_EN_pinyin/vocab.txt (minimal, hanya yang dibutuhkan prepare_csv_wavs.py)
EMILIA_VOCAB_DIR = REPO_DIR / "data" / "Emilia_ZH_EN_pinyin"
EMILIA_VOCAB_PATH = EMILIA_VOCAB_DIR / "vocab.txt"
EMILIA_VOCAB_DIR.mkdir(parents=True, exist_ok=True)

if EMILIA_VOCAB_PATH.exists() and EMILIA_VOCAB_PATH.stat().st_size > 0:
    print("Pretrained vocab sudah ada:", EMILIA_VOCAB_PATH)
else:
    run_cmd(["uv", "pip", "install", "--python", str(VENV_PY), "-U", "huggingface_hub"])

    vocab_fetch_script = "\n".join([
        "from pathlib import Path",
        "import shutil",
        "from huggingface_hub import hf_hub_download",
        f"target = Path(r'{EMILIA_VOCAB_PATH}')",
        "target.parent.mkdir(parents=True, exist_ok=True)",
        "candidates = [",
        "    ('SWivid/F5-TTS', 'F5TTS_Base/vocab.txt'),",
        "    ('SWivid/F5-TTS', 'F5TTS_v1_Base/vocab.txt'),",
        "]",
        "last_err = None",
        "for repo_id, filename in candidates:",
        "    try:",
        "        src = Path(hf_hub_download(repo_id=repo_id, filename=filename))",
        "        shutil.copy2(src, target)",
        "        print(f'Downloaded vocab from {repo_id}/{filename} -> {target}')",
        "        break",
        "    except Exception as e:",
        "        print(f'Gagal dari {repo_id}/{filename}: {e}')",
        "        last_err = e",
        "else:",
        "    raise RuntimeError(f'Gagal download vocab Emilia_ZH_EN_pinyin: {last_err}')",
        "print('vocab exists:', target.exists(), 'size:', target.stat().st_size if target.exists() else -1)",
    ])

    run_py(["-c", vocab_fetch_script], cwd=REPO_DIR)

run_cmd(["ls", "-lah", str(EMILIA_VOCAB_DIR)])

In [ ]:
# 7) Jalankan prepare_csv_wavs.py di notebook (Kaggle-aware vocab detect)
PREPARED_DATASET_DIR.mkdir(parents=True, exist_ok=True)

expected_vocab = REPO_DIR / "data" / "Emilia_ZH_EN_pinyin" / "vocab.txt"
if not expected_vocab.exists() or expected_vocab.stat().st_size == 0:
    data_root = REPO_DIR / "data"
    fallback = None
    for cand in data_root.glob("**/vocab.txt"):
        if cand.is_file() and cand.stat().st_size > 0:
            fallback = cand
            break

    if fallback is None:
        raise FileNotFoundError(
            f"vocab.txt tidak ditemukan untuk finetune prepare. Expected: {expected_vocab}"
        )

    expected_vocab.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(fallback, expected_vocab)
    print("Fallback vocab dipakai:", fallback, "->", expected_vocab)

print("Using pretrained vocab:", expected_vocab)

# Self-heal jika setup cell tidak dijalankan ulang setelah perubahan dependency profile.
try:
    run_py(["-c", "import f5_tts; print('f5_tts import ok')"], cwd=REPO_DIR)
except subprocess.CalledProcessError:
    print("f5_tts belum terinstall di venv, install editable package...")
    run_cmd([
        "uv",
        "pip",
        "install",
        "--python",
        str(VENV_PY),
        "-e",
        str(REPO_DIR),
    ], cwd=REPO_DIR)

run_py([
    "src/f5_tts/train/datasets/prepare_csv_wavs.py",
    str(MERGED_CSV),
    str(PREPARED_DATASET_DIR),
    "--workers",
    "8",
], cwd=REPO_DIR)

run_cmd(["ls", "-lah", str(PREPARED_DATASET_DIR)])

In [ ]:
# 8) W&B hardcoded login + sanity logging
env = os.environ.copy()
env["WANDB_API_KEY"] = WANDB_API_KEY
env["WANDB_ENTITY"] = WANDB_ENTITY
env["WANDB_PROJECT"] = WANDB_PROJECT

wandb_smoke = """
import os
import random
import wandb

api_key = os.environ["WANDB_API_KEY"]
entity = os.environ["WANDB_ENTITY"]
project = os.environ.get("WANDB_PROJECT", "kaceve")

wandb.login(key=api_key)
run = wandb.init(
    entity=entity,
    project=project,
    config={
        "learning_rate": 0.02,
        "architecture": "CNN",
        "dataset": "CIFAR-100",
        "epochs": 10,
    },
)
epochs = 10
offset = random.random() / 5
for epoch in range(2, epochs):
    acc = 1 - 2**-epoch - random.random() / epoch - offset
    loss = 2**-epoch + random.random() / epoch + offset
    run.log({"acc": acc, "loss": loss})
run.finish()
print("wandb sanity done")
""".strip()

run_py(["-c", wandb_smoke], cwd=REPO_DIR, env=env)

In [ ]:
# 9) Persiapan runtime training + mamba (tanpa menjalankan training)
env = os.environ.copy()
env["WANDB_API_KEY"] = WANDB_API_KEY
env["WANDB_ENTITY"] = WANDB_ENTITY

CKPT_ROOT_WORKING = Path("/kaggle/working/ckpts")
DISTILL_TAG = "distill_final_datasetku"
FULL_TAG = "full_final_datasetku"

distill_dir_abs = CKPT_ROOT_WORKING / DISTILL_TAG
full_dir_abs = CKPT_ROOT_WORKING / FULL_TAG
distill_dir_abs.mkdir(parents=True, exist_ok=True)
full_dir_abs.mkdir(parents=True, exist_ok=True)

# train.py menyimpan checkpoint ke repo_root/ckpts.save_dir, jadi kita pakai relative escape ke /kaggle/working
distill_save_dir_rel = f"../../working/ckpts/{DISTILL_TAG}"
full_save_dir_rel = f"../../working/ckpts/{FULL_TAG}"

no_periodic_ckpt_overrides = [
    "ckpts.save_per_updates=999999999",
    "ckpts.last_per_updates=999999999",
    "ckpts.keep_last_n_checkpoints=0",
    "ckpts.log_samples=False",
]

MAMBA_PROBE_TIMEOUT_SEC = 90
MAMBA_POST_REPAIR_TIMEOUT_SEC = 240

def _build_runtime_env(base_env):
    runtime_env = base_env.copy()
    site_pkgs_candidates = sorted((VENV_DIR / "lib").glob("python*/site-packages"))
    if not site_pkgs_candidates:
        return runtime_env

    sp = site_pkgs_candidates[-1]
    cuda_lib_rels = [
        "nvidia/cublas/lib",
        "nvidia/cuda_runtime/lib",
        "nvidia/cudnn/lib",
        "nvidia/cufft/lib",
        "nvidia/curand/lib",
        "nvidia/cusolver/lib",
        "nvidia/cusparse/lib",
        "nvidia/nccl/lib",
        "nvidia/nvjitlink/lib",
    ]
    cuda_libs = [str(sp / rel) for rel in cuda_lib_rels if (sp / rel).exists()]
    if cuda_libs:
        current = runtime_env.get("LD_LIBRARY_PATH", "")
        runtime_env["LD_LIBRARY_PATH"] = ":".join(cuda_libs + ([current] if current else []))
        print("LD_LIBRARY_PATH prepared for CUDA libs in venv.")

    runtime_env.setdefault("PYTHONFAULTHANDLER", "1")
    return runtime_env

runtime_env = _build_runtime_env(env)

def run_py_nosync(args, cwd=None, timeout=None):
    return run_cmd(
        ["uv", "run", "--no-sync", "--python", str(VENV_PY), "python", *args],
        cwd=cwd,
        env=runtime_env,
        timeout=timeout,
    )

def _probe_mamba(timeout_sec=MAMBA_PROBE_TIMEOUT_SEC) -> None:
    probe_script = "\n".join([
        "print('probe: import torch...', flush=True)",
        "import torch",
        "print('probe: torch =', torch.__version__, 'cuda =', torch.version.cuda, flush=True)",
        "print('probe: import mamba_ssm...', flush=True)",
        "import mamba_ssm",
        "print('probe: import selective_scan_cuda...', flush=True)",
        "import selective_scan_cuda",
        "print('mamba probe ok', flush=True)",
    ])
    run_py_nosync(["-u", "-X", "faulthandler", "-c", probe_script], cwd=REPO_DIR, timeout=timeout_sec)

def _ensure_torch_compat() -> None:
    check_script = "\n".join([
        "import torch",
        "print('torch =', torch.__version__, 'cuda =', torch.version.cuda)",
        "ok = torch.__version__.startswith('2.8.0') and str(torch.version.cuda).startswith('12.8')",
        "raise SystemExit(0 if ok else 1)",
    ])
    try:
        run_py_nosync(["-c", check_script], cwd=REPO_DIR, timeout=60)
        return
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        print("Torch stack tidak cocok, paksa reinstall ke torch 2.8.0+cu128.")

    run_cmd([
        "uv",
        "pip",
        "install",
        "--python",
        str(VENV_PY),
        "--index-url",
        "https://download.pytorch.org/whl/cu128",
        "--extra-index-url",
        "https://pypi.org/simple",
        "--index-strategy",
        "unsafe-best-match",
        "--force-reinstall",
        "--no-cache-dir",
        "torch==2.8.0+cu128",
        "torchvision==0.23.0+cu128",
        "torchaudio==2.8.0+cu128",
        "nvidia-nccl-cu12==2.27.3",
        "nvidia-nvjitlink-cu12==12.8.93",
    ], cwd=REPO_DIR)

    run_py_nosync(["-c", check_script], cwd=REPO_DIR, timeout=60)

def _ensure_python_headers() -> None:
    py_mm = subprocess.check_output(
        [str(VENV_PY), "-c", "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')"],
        text=True,
    ).strip()
    py_header = Path(f"/usr/include/python{py_mm}/Python.h")
    if py_header.exists():
        return

    print(f"Python headers tidak ditemukan ({py_header}), install python{py_mm}-dev...")
    run_cmd(["apt-get", "update", "-y"])
    run_cmd(["apt-get", "install", "-y", f"python{py_mm}-dev", "build-essential"])

    if not py_header.exists():
        raise FileNotFoundError(f"Python.h tetap tidak ditemukan di {py_header}")

def _repair_mamba() -> None:
    print("Repair mamba dimulai...")
    _ensure_torch_compat()
    _ensure_python_headers()

    run_cmd(
        [str(VENV_PY), "-m", "pip", "install", "--upgrade", "pip", "wheel", "ninja", "setuptools<82"],
        cwd=REPO_DIR,
        env=runtime_env,
    )

    wheel_env = runtime_env.copy()
    wheel_env["TORCH_CUDA_ARCH_LIST"] = "7.5"
    wheel_env["MAX_JOBS"] = "2"

    wheel_cmd = [
        "uv",
        "pip",
        "install",
        "--python",
        str(VENV_PY),
        "--index-url",
        "https://pypi.org/simple",
        "--extra-index-url",
        "https://download.pytorch.org/whl/cu128",
        "--index-strategy",
        "unsafe-best-match",
        "--force-reinstall",
        "--no-cache-dir",
        "--prefer-binary",
        "--no-build-isolation",
        "--no-deps",
        "causal-conv1d",
        "mamba-ssm",
    ]

    try:
        run_cmd(wheel_cmd, cwd=REPO_DIR, env=wheel_env)
        _probe_mamba(timeout_sec=120)
        print("Mamba wheel kompatibel.")
        return
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        print("Wheel mamba gagal/timeout, lanjut build from source...")

    build_env = wheel_env.copy()
    build_env["MAMBA_FORCE_BUILD"] = "TRUE"
    build_env["CAUSAL_CONV1D_FORCE_BUILD"] = "TRUE"

    run_cmd(
        [
            str(VENV_PY),
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--no-build-isolation",
            "--force-reinstall",
            "--no-binary",
            ":all:",
            "--no-deps",
            "causal-conv1d",
        ],
        cwd=REPO_DIR,
        env=build_env,
    )
    run_cmd(
        [
            str(VENV_PY),
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--no-build-isolation",
            "--force-reinstall",
            "--no-binary",
            ":all:",
            "--no-deps",
            "mamba-ssm",
        ],
        cwd=REPO_DIR,
        env=build_env,
    )

    _probe_mamba(timeout_sec=MAMBA_POST_REPAIR_TIMEOUT_SEC)
    print("Mamba berhasil dibangun dari source.")

try:
    _probe_mamba()
    print("mamba_mode: enabled (fast-path)")
except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
    print("Probe mamba gagal/timeout. Menjalankan repair...")
    _repair_mamba()
    _probe_mamba(timeout_sec=MAMBA_POST_REPAIR_TIMEOUT_SEC)
    print("mamba_mode: enabled (after repair)")

print("Persiapan selesai. Lanjut ke cell distill.")

In [ ]:
# 10) HOTFIX runtime (sementara): patch checkpoint-compat di trainer.py
from pathlib import Path

trainer_path = REPO_DIR / "src/f5_tts/model/trainer.py"
src = trainer_path.read_text(encoding="utf-8")

if "Checkpoint loaded in compatibility mode; update reset to 0" in src:
    print("Hotfix sudah ada di trainer.py")
else:
    # Gunakan string literal dengan indent asli supaya patch aman untuk runtime process lain.
    old_ema_block = (
        "        if self.is_main:\n"
        "            self.ema_model.load_state_dict(checkpoint[\"ema_model_state_dict\"])\n"
    )
    new_ema_block = (
        "        ema_loaded_strict = True\n"
        "        if self.is_main:\n"
        "            try:\n"
        "                self.ema_model.load_state_dict(checkpoint[\"ema_model_state_dict\"])\n"
        "            except RuntimeError as exc:\n"
        "                ema_loaded_strict = False\n"
        "                print(f\"Strict EMA checkpoint load failed, retrying non-strict for compatibility: {exc}\")\n"
        "                self.ema_model.load_state_dict(checkpoint[\"ema_model_state_dict\"], strict=False)\n"
    )

    old_resume_block = (
        "            try:\n"
        "                self.accelerator.unwrap_model(self.model).load_state_dict(checkpoint[\"model_state_dict\"])\n"
        "            except RuntimeError as exc:\n"
        "                print(f\"Strict checkpoint load failed, retrying non-strict for compatibility: {exc}\")\n"
        "                self.accelerator.unwrap_model(self.model).load_state_dict(checkpoint[\"model_state_dict\"], strict=False)\n"
        "            self.optimizer.load_state_dict(checkpoint[\"optimizer_state_dict\"])\n"
        "            if self.scheduler:\n"
        "                self.scheduler.load_state_dict(checkpoint[\"scheduler_state_dict\"])\n"
        "            update = checkpoint[\"update\"]\n"
    )

    new_resume_block = (
        "            model_loaded_strict = True\n"
        "            try:\n"
        "                self.accelerator.unwrap_model(self.model).load_state_dict(checkpoint[\"model_state_dict\"])\n"
        "            except RuntimeError as exc:\n"
        "                model_loaded_strict = False\n"
        "                print(f\"Strict checkpoint load failed, retrying non-strict for compatibility: {exc}\")\n"
        "                self.accelerator.unwrap_model(self.model).load_state_dict(checkpoint[\"model_state_dict\"], strict=False)\n"
        "\n"
        "            optimizer_loaded = True\n"
        "            try:\n"
        "                self.optimizer.load_state_dict(checkpoint[\"optimizer_state_dict\"])\n"
        "            except Exception as exc:\n"
        "                optimizer_loaded = False\n"
        "                print(f\"Optimizer checkpoint load failed, using fresh optimizer state: {exc}\")\n"
        "\n"
        "            scheduler_loaded = True\n"
        "            if self.scheduler:\n"
        "                try:\n"
        "                    self.scheduler.load_state_dict(checkpoint[\"scheduler_state_dict\"])\n"
        "                except Exception as exc:\n"
        "                    scheduler_loaded = False\n"
        "                    print(f\"Scheduler checkpoint load failed, using fresh scheduler state: {exc}\")\n"
        "\n"
        "            if model_loaded_strict and optimizer_loaded and scheduler_loaded and ema_loaded_strict:\n"
        "                update = checkpoint[\"update\"]\n"
        "            else:\n"
        "                update = 0\n"
        "                print(\n"
        "                    \"Checkpoint loaded in compatibility mode; update reset to 0 \"\n"
        "                    \"to avoid resuming with incompatible optimizer/scheduler state.\"\n"
        "                )\n"
    )

    missing = []
    if old_ema_block in src:
        src = src.replace(old_ema_block, new_ema_block, 1)
    elif new_ema_block in src:
        pass
    else:
        missing.append("EMA load block")

    if old_resume_block in src:
        src = src.replace(old_resume_block, new_resume_block, 1)
    elif new_resume_block in src:
        pass
    else:
        missing.append("resume block")

    if missing:
        raise RuntimeError(
            "Gagal patch trainer.py, blok tidak ditemukan: " + ", ".join(missing)
        )

    trainer_path.write_text(src, encoding="utf-8")
    print("Hotfix berhasil diterapkan:", trainer_path)

run_py(["-m", "py_compile", "src/f5_tts/model/trainer.py"], cwd=REPO_DIR)
print("trainer.py syntax OK")
run_py([
    "-c",
    "from f5_tts.model.trainer import Trainer; print('trainer import OK after hotfix')",
], cwd=REPO_DIR)

In [ ]:
# 10) Distill phase (final checkpoint only)
distill_run_name = "F5TTS_HYBRID_DISTILL_5K_datasetku_kaggle_2xT4"

distill_cmd = [
    "uv",
    "run",
    "--no-sync",
    "--python",
    str(VENV_PY),
    "accelerate",
    "launch",
    "--num_processes=2",
    "--mixed_precision=fp16",
    "src/f5_tts/train/train.py",
    "--config-name",
    "F5TTS_HYBRID_DISTILL_5K.yaml",
    "datasets.name=datasetku",
    f"model.cfm_experiment.teacher_ckpt_path={TEACHER_CKPT}",
    "datasets.batch_size_per_gpu=768",
    "datasets.max_samples=1",
    "datasets.num_workers=4",
    "optim.epochs=1",
    "optim.num_warmup_updates=500",
    "ckpts.logger=wandb",
    f"ckpts.wandb_project={WANDB_PROJECT}",
    f"ckpts.wandb_run_name={distill_run_name}",
    f"ckpts.save_dir={distill_save_dir_rel}",
    *no_periodic_ckpt_overrides,
]

run_cmd(distill_cmd, cwd=REPO_DIR, env=runtime_env)
run_cmd(["ls", "-lah", str(distill_dir_abs)])

distill_last = distill_dir_abs / "model_last.pt"
if not distill_last.exists():
    raise FileNotFoundError(f"Distill final checkpoint tidak ditemukan: {distill_last}")

# full training BUKAN dari nol: partial init dari parent checkpoint yang sama dengan teacher distill
if not TEACHER_CKPT.exists():
    raise FileNotFoundError(f"Parent checkpoint tidak ditemukan: {TEACHER_CKPT}")

pretrained_for_full = full_dir_abs / "pretrained_indo_v2.pt"
shutil.copy2(TEACHER_CKPT, pretrained_for_full)

print("Distill final checkpoint:", distill_last)
print("Prepared full-training partial init checkpoint (same parent as distill):", pretrained_for_full)
print("Distill selesai. Lanjut ke cell full training.")

In [ ]:
# 11) Full training phase (final checkpoint only)
pretrained_for_full = full_dir_abs / "pretrained_indo_v2.pt"
if not pretrained_for_full.exists():
    raise FileNotFoundError(
        f"Partial init checkpoint untuk full training belum ada: {pretrained_for_full}. Jalankan cell distill dulu."
    )

full_run_name = "F5TTS_EarlyBiMamba_v1_datasetku_kaggle_2xT4"

full_cmd = [
    "uv",
    "run",
    "--no-sync",
    "--python",
    str(VENV_PY),
    "accelerate",
    "launch",
    "--num_processes=2",
    "--mixed_precision=fp16",
    "src/f5_tts/train/train.py",
    "--config-name",
    "F5TTS_EarlyBiMamba_v1.yaml",
    "datasets.name=datasetku",
    "datasets.batch_size_per_gpu=768",
    "datasets.max_samples=1",
    "datasets.num_workers=4",
    "optim.epochs=1",
    "optim.num_warmup_updates=500",
    "ckpts.logger=wandb",
    f"ckpts.wandb_project={WANDB_PROJECT}",
    f"ckpts.wandb_run_name={full_run_name}",
    f"ckpts.save_dir={full_save_dir_rel}",
    *no_periodic_ckpt_overrides,
]

run_cmd(full_cmd, cwd=REPO_DIR, env=runtime_env)
run_cmd(["ls", "-lah", str(full_dir_abs)])

full_last = full_dir_abs / "model_last.pt"
if not full_last.exists():
    raise FileNotFoundError(f"Full-training final checkpoint tidak ditemukan: {full_last}")

distill_last = distill_dir_abs / "model_last.pt"
if not distill_last.exists():
    raise FileNotFoundError(f"Distill final checkpoint tidak ditemukan: {distill_last}")

print("\nFinal checkpoints:")
print("- Distill final:", distill_last)
print("- Full final:", full_last)

In [ ]:
# 12) Inference test: ref_prabowo.mp3 dari Eempostor/F5-TTS-INDO-FINETUNE-V2
from pathlib import Path
from IPython.display import Audio, display

REF_REPO_ID = "Eempostor/F5-TTS-INDO-FINETUNE-V2"
REF_FILENAME = "ref_prabowo.mp3"
REF_TEXT = "SELAMAT PAGI SALAM SEJAHTERA BAGI KITA SEMUA"
# Untuk uji cepat, target teks disamakan dengan transkrip referensi. Ubah baris ini kalau mau teks lain.
GEN_TEXT = REF_TEXT
INFER_OUT_WAV = Path("/kaggle/working/infer_ref_prabowo_test.wav")

if "runtime_env" not in globals():
    runtime_env = os.environ.copy()

# Pastikan dependency download HF tersedia di venv runtime.
run_cmd(["uv", "pip", "install", "--python", str(VENV_PY), "-U", "huggingface_hub"], cwd=REPO_DIR)

infer_script = f"""
from pathlib import Path
from huggingface_hub import hf_hub_download
from f5_tts.api import F5TTS

repo_id = {REF_REPO_ID!r}
ref_audio = hf_hub_download(
    repo_id=repo_id,
    filename={REF_FILENAME!r},
    local_dir="/kaggle/working/hf_refs",
    local_dir_use_symlinks=False,
)

vocab_file = {str(REPO_DIR / "data/Emilia_ZH_EN_pinyin/vocab.txt")!r}
ckpt_candidates = [
    ("F5TTS_EarlyBiMamba_v1", Path("/kaggle/working/ckpts/full_final_datasetku/model_last.pt")),
    ("F5TTS_HYBRID_DISTILL_5K", Path("/kaggle/working/ckpts/distill_final_datasetku/model_last.pt")),
]

model_name = None
ckpt_file = None
for name, ckpt in ckpt_candidates:
    if ckpt.exists():
        model_name = name
        ckpt_file = str(ckpt)
        break

if ckpt_file is None:
    raise FileNotFoundError("Checkpoint infer tidak ditemukan. Jalankan distill/full training dulu.")

tts = F5TTS(
    model=model_name,
    ckpt_file=ckpt_file,
    vocab_file=vocab_file,
    use_ema=True,
    device=None,
)

tts.infer(
    ref_file=ref_audio,
    ref_text={REF_TEXT!r},
    gen_text={GEN_TEXT!r},
    file_wave={str(INFER_OUT_WAV)!r},
    nfe_step=32,
    speed=1.0,
    seed=1234,
)

print("Inference selesai")
print("model =", model_name)
print("ckpt =", ckpt_file)
print("ref_audio =", ref_audio)
print("out_wav =", {str(INFER_OUT_WAV)!r})
""".strip()

run_py(["-c", infer_script], cwd=REPO_DIR, env=runtime_env)

if not INFER_OUT_WAV.exists():
    raise FileNotFoundError(f"Output audio tidak ditemukan: {INFER_OUT_WAV}")

print("Saved:", INFER_OUT_WAV)
display(Audio(str(INFER_OUT_WAV)))

In [ ]:
# 12) Inference test: ref_prabowo.mp3 dari Eempostor/F5-TTS-INDO-FINETUNE-V2 (1xGPU, drop-in, auto device, dtype fix)
from pathlib import Path
from IPython.display import Audio, display

REF_REPO_ID   = "Eempostor/F5-TTS-INDO-FINETUNE-V2"
REF_FILENAME  = "ref_prabowo.mp3"
REF_TEXT      = "SELAMAT PAGI SALAM SEJAHTERA BAGI KITA SEMUA"
GEN_TEXT      = REF_TEXT
INFER_OUT_WAV = Path("/kaggle/working/infer_ref_prabowo_test.wav")

if "runtime_env" not in globals():
    runtime_env = os.environ.copy()

# ✅ Fix: override Jupyter's inline backend with a headless one
runtime_env["MPLBACKEND"] = "Agg"

# Pastikan dependency download HF tersedia di venv runtime.
run_cmd(["uv", "pip", "install", "--python", str(VENV_PY), "-U", "huggingface_hub"], cwd=REPO_DIR)

infer_script = f"""
from pathlib import Path
from huggingface_hub import hf_hub_download
from f5_tts.api import F5TTS
import torch

repo_id   = {REF_REPO_ID!r}
ref_audio = hf_hub_download(
    repo_id=repo_id,
    filename={REF_FILENAME!r},
    local_dir="/kaggle/working/hf_refs",
    local_dir_use_symlinks=False,
)

vocab_file = {str(REPO_DIR / "data/Emilia_ZH_EN_pinyin/vocab.txt")!r}
ckpt_candidates = [
    ("F5TTS_EarlyBiMamba_v1",   Path("/kaggle/working/ckpts/full_final_datasetku/model_last.pt")),
    ("F5TTS_HYBRID_DISTILL_5K", Path("/kaggle/working/ckpts/distill_final_datasetku/model_last.pt")),
]

model_name = None
ckpt_file  = None
for name, ckpt in ckpt_candidates:
    if ckpt.exists():
        model_name = name
        ckpt_file  = str(ckpt)
        break

if ckpt_file is None:
    raise FileNotFoundError("Checkpoint infer tidak ditemukan. Jalankan distill/full training dulu.")

# Pilih device otomatis (cuda jika ada, fallback cpu)
device = "cuda" if torch.cuda.is_available() else "cpu"

tts = F5TTS(
    model=model_name,
    ckpt_file=ckpt_file,
    vocab_file=vocab_file,
    use_ema=True,
    device=device,
)
# Cast semua model ke half jika pakai CUDA (untuk mixed precision)
if device == "cuda":
    if hasattr(tts, "ema_model") and tts.ema_model is not None:
        tts.ema_model.half()
    if hasattr(tts, "model") and tts.model is not None:
        tts.model.half()

tts.infer(
    ref_file=ref_audio,
    ref_text={REF_TEXT!r},
    gen_text={GEN_TEXT!r},
    file_wave={str(INFER_OUT_WAV)!r},
    nfe_step=32,
    speed=1.0,
    seed=1234,
)

print("Inference selesai")
print("model =", model_name)
print("ckpt  =", ckpt_file)
print("ref   =", ref_audio)
print("out   =", {str(INFER_OUT_WAV)!r})
""".strip()

run_py(["-c", infer_script], cwd=REPO_DIR, env=runtime_env)

if not INFER_OUT_WAV.exists():
    raise FileNotFoundError(f"Output audio tidak ditemukan: {INFER_OUT_WAV}")

print("Saved:", INFER_OUT_WAV)
display(Audio(str(INFER_OUT_WAV)))